[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C23_Frontier_Alignment_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与对齐方法论热身

本课全程 **纯 numpy、CPU 可跑**，用**玩具模型 + 合成偏好**模拟前沿对齐方法的*逻辑结构*，再用 `assert` 验证正确性。

这个 notebook 做四件事：① 确认环境；② 用最小例子体会 **RLHF 的核心数学（Bradley-Terry 偏好）**；③ 亲眼看见 **代理目标 ≠ 真目标（Goodhart）** 的雏形；④ 立下全课的纪律——**在已知真相的沙盒里用 `assert` 校验机制**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅  本课无需 GPU/联网/大模型')

## 2 · RLHF 的心脏：Bradley-Terry 偏好模型

RLHF 把「人类觉得 A 比 B 好」建模成一个概率：两个回答各有一个潜在「质量分」$r_A, r_B$，则

$$P(A \succ B) = \sigma(r_A - r_B), \qquad \sigma(z)=\frac{1}{1+e^{-z}}$$

偏好概率**只取决于分差**。分差越大越笃定，分差为 0 则五五开。这是模块 02 训练奖励模型的基础，先把它跑通。

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def bt_prob(r_a, r_b):
    '''Bradley-Terry: A 优于 B 的概率 = sigmoid(分差)。'''
    return sigmoid(r_a - r_b)

# 分差为 0 -> 0.5；A 明显更好 -> 接近 1
print('r_A=r_B   :', bt_prob(2.0, 2.0))
print('A 高 2 分 :', round(float(bt_prob(3.0, 1.0)), 4))
print('A 低 2 分 :', round(float(bt_prob(1.0, 3.0)), 4))
assert abs(bt_prob(2.0, 2.0) - 0.5) < 1e-12, '分差为0应为0.5'
assert bt_prob(3.0, 1.0) > 0.5 > bt_prob(1.0, 3.0), '分高者更可能被偏好'
# 互补性：P(A≻B) + P(B≻A) = 1
assert abs(bt_prob(3.0, 1.0) + bt_prob(1.0, 3.0) - 1.0) < 1e-12
# 只依赖分差：整体加常数不变
assert abs(bt_prob(3.0, 1.0) - bt_prob(13.0, 11.0)) < 1e-12
print('✅ Bradley-Terry 性质成立：单调、互补、只看分差')

**关键结论**：偏好概率是分差的 sigmoid。训练奖励模型（模块 02）就是反过来——给定一堆观测到的偏好，调 $r$ 使这些偏好在 BT 模型下的似然最大。这把「主观偏好」变成了「可优化的数值目标」。

## 3 · 从偏好合成数据：我们的「人类标注」长这样

本课没有真人标注。我们设一个 **gold 质量函数**（=我们假装的「真实人类偏好」），按 Bradley-Terry **概率性地**给每对回答标 chosen/rejected。
这样我们**知道全部真相**，能检验任何方法学到的奖励是否逼近 gold —— 这是全课沙盒的基石。

In [ ]:
rng = np.random.default_rng(0)

def gold_quality(x):
    '''假装的「真实质量」：这里用一个简单线性函数当 ground truth。'''
    return x @ np.array([1.0, -0.5, 0.3])

def make_preference(a, b, rng):
    '''按 Bradley-Terry 概率标注：以 P=σ(gold_a−gold_b) 判 a 优于 b。'''
    pa = bt_prob(gold_quality(a), gold_quality(b))
    a_wins = rng.random() < pa
    return (a, b) if a_wins else (b, a)   # 返回 (chosen, rejected)

# 造 2000 对，统计「chosen 的 gold 分确实更高」的比例
n = 2000
wins = 0
for _ in range(n):
    a = rng.standard_normal(3); b = rng.standard_normal(3)
    chosen, rejected = make_preference(a, b, rng)
    if gold_quality(chosen) > gold_quality(rejected):
        wins += 1
frac = wins / n
print(f'chosen 的 gold 分更高的比例 = {frac:.3f}')
# 偏好带噪（σ 不是阶跃），但应显著优于随机 0.5
assert 0.6 < frac < 0.85, '合成偏好应与 gold 相关但带噪'
print('✅ 合成偏好与 gold 质量相关、但有噪声 —— 正如真实人类标注')

## 4 · 一眼看穿 Goodhart：代理目标 ≠ 真目标

对齐最核心的失效模式：我们优化的是**代理**（proxy，如奖励模型），想要的是**真目标**（gold）。若代理与真目标只是**相关而非相等**，把代理优化到极致往往让真目标先升后**降**。

用一个最小例子演示：gold 想要 `x≈3`，但我们的 proxy 错误地「越大越好」。沿 proxy 爬坡，看 gold 怎么先升后降。

In [ ]:
def gold_obj(x):
    return -(x - 3.0) ** 2          # 真目标：x=3 最好（倒抛物线）

def proxy_obj(x):
    return x                        # 代理：错误地以为越大越好

xs = np.linspace(0, 8, 17)
print(f"{'x':>5} {'proxy':>8} {'gold':>8}")
for x in xs:
    print(f'{x:>5.1f} {proxy_obj(x):>8.2f} {gold_obj(x):>8.2f}')
# proxy 单调升；gold 在 x=3 见顶后下降
proxy_vals = proxy_obj(xs); gold_vals = gold_obj(xs)
assert np.all(np.diff(proxy_vals) > 0), 'proxy 单调递增'
best_x = xs[np.argmax(gold_vals)]
assert abs(best_x - 3.0) < 0.6, 'gold 的峰应在 x≈3'
assert gold_vals[-1] < gold_vals[np.argmax(gold_vals)], '继续优化 proxy 会让 gold 下降'
print('\n✅ proxy 一路升，gold 过 x≈3 后掉头向下 —— 这就是过优化/Goodhart 的骨架（模块 02 展开）')

## 5 · 立纪律：在「已知真相的沙盒」里用 assert 校验

本课每个机制都设一个我们**知道 ground truth** 的合成沙盒（埋入的真概念、设定的 gold 奖励、给定的真有害标签），再用 `assert` 检查方法的行为是否符合预期。把这个判定封装成一个小工具，后面每个模块都用它。

In [ ]:
def check(name, cond, detail=''):
    '''统一的沙盒断言：打印结果并 assert。'''
    status = '✅' if cond else '❌'
    print(f'[{status}] {name}' + (f'  ({detail})' if detail else ''))
    assert cond, f'{name} 不成立！{detail}'
    return cond

# 演示：用它复述前面几条结论
check('Bradley-Terry 单调', bt_prob(3.0, 1.0) > bt_prob(2.0, 1.0))
check('合成偏好优于随机', frac > 0.5, detail=f'frac={frac:.3f}')
check('proxy 优化致 gold 下降', gold_vals[-1] < gold_vals.max())
print('\n这就是全课的工作流：设沙盒(知真相) -> 跑机制 -> assert 兜底。')

## 6 · 对齐方法地图：本课五招与它们补的「裂缝」

把五个模块和它们在 RLHF 流水线上修补的位置列成一张表，作为全课的导航。

In [ ]:
ALIGN_MAP = [
    # (模块, 方法, 补的裂缝, 一句话机制)
    ('01', 'Constitutional AI / RLAIF', '标注瓶颈',   '人写宪法, AI 自我批判-修订 + AI 标注偏好'),
    ('02', '奖励建模与过优化',        '奖励失真',   'Bradley-Terry 训 RM; 集成分歧抗 hacking'),
    ('03', '可扩展监督 / debate',      '监督天花板', '弱裁判 + 强论证/分解, 揭穿>欺骗'),
    ('04', 'Weak-to-Strong',          '监督天花板', '弱标签训强学生, 引出潜在能力(PGR)'),
    ('05', 'Deliberative Alignment',   '监督天花板', '回答前显式推理明文规范(spec)'),
]
print(f"{'模块':<4}{'方法':<28}{'裂缝':<10}{'机制':<40}")
for num, method, gap, mech in ALIGN_MAP:
    print(f'{num:<4}{method:<28}{gap:<10}{mech:<40}')
assert len(ALIGN_MAP) == 5
gaps = {g for _, _, g, _ in ALIGN_MAP}
assert gaps == {'标注瓶颈', '奖励失真', '监督天花板'}, '应覆盖三道裂缝'
print('\n✅ 五个模块覆盖 RLHF 的三道裂缝；03–05 共同攻「监督天花板」这一终极难题。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：每个对齐方法都在一个我们*知道真相*的 numpy 沙盒里实现，并用 `assert` 验证其机制成立；机制正确则逻辑可迁移到真实 RLHF/RLAIF 管线。

**接下来五个模块**：01 Constitutional AI → 02 奖励建模与过优化 → 03 可扩展监督 → 04 weak-to-strong → 05 deliberative。它们沿「监督信号从哪来」这条主线层层递进。

下一站：**模块 01 · Constitutional AI 与 RLAIF**。